In [15]:
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent
print(f"Project root: {PROJECT_ROOT}")
DATA_DIR = PROJECT_ROOT / "data"

def read_json(file_path: Path | str) -> dict | list:
    data = None
    with open(file_path, "r") as f:
        data = json.load(f)
    return data

def read_jsonl(file_path: Path | str) -> list:
    res = []
    with open(file_path, "r") as f:
        for line in f:
            res.append(json.loads(line))
    return res

def write_json(data: dict | list, file_path: Path | str) -> None:
    with open(file_path, "w") as f:
        json.dump(data, f, indent=4)
        print(f"wrote to {file_path}")

Project root: /Users/matthewho/Documents/research/ctx_editor


In [8]:
original_lic_data_path = DATA_DIR / "sharded_instructions_600.json"
original_lic_data = read_json(original_lic_data_path)

In [10]:
# get task == "code" and task_id contains "livecodebench"
lic_lcb = [item for item in original_lic_data if item.get("task") == "code" and "livecodebench" in item.get("task_id", "")]

In [3]:
bstdoutp = PROJECT_ROOT / "baseline_stdout.txt"
txt = bstdoutp.read_text()

In [4]:
pattern = "Error evaluating SQL: tuple index out of range"

In [5]:
# count how many times this pattern appears in the baseline stdout
count = txt.count(pattern)
print(f"The pattern '{pattern}' appears {count} times in the baseline stdout.")

The pattern 'Error evaluating SQL: tuple index out of range' appears 55 times in the baseline stdout.


## hard task LiC run prep

In [20]:
# prep the 1 task per thing san check
aime24_path = PROJECT_ROOT / "data/aime24_sharded_m8.json"
lcbh_path = PROJECT_ROOT / "data/lcbh_exp_data.json"
aime24 = read_json(aime24_path)
lcbh = read_json(lcbh_path)

print(len(aime24))
print(len(lcbh))

30
29


In [21]:
print(aime24[0].keys())
print(lcbh[0].keys())

dict_keys(['question', 'full_spec_q', 'shards', '_segments', 'answer', 'ground_truth_a', 'task_id', 'task'])
dict_keys(['question_title', 'question_content', 'platform', 'question_id', 'contest_id', 'contest_date', 'starter_code', 'difficulty', 'public_test_cases', 'private_test_cases', 'metadata', 'task_id', 'source', 'shards', 'task'])


In [22]:
lic_lcb[0].keys()

dict_keys(['question_title', 'question_content', 'platform', 'question_id', 'contest_id', 'contest_date', 'starter_code', 'difficulty', 'public_test_cases', 'private_test_cases', 'metadata', 'task_id', 'source', 'shards', 'task'])

In [23]:
lcbh[0]["metadata"]

{'func_name': 'maxSubarraySum'}

In [24]:
# ok now let's create a dummy json
hard_san_check_data = [
    aime24[0],
    lcbh[0],
]
hard_san_check_path = DATA_DIR / "hard_task_san_check.json"
write_json(data=hard_san_check_data, file_path=hard_san_check_path)

wrote to /Users/matthewho/Documents/research/ctx_editor/data/hard_task_san_check.json


In [25]:
# ok now the real merge
hard_task_data = aime24 + lcbh
hard_task_path = DATA_DIR / "hard_task_exp_data.json"
write_json(data=hard_task_data, file_path=hard_task_path)

wrote to /Users/matthewho/Documents/research/ctx_editor/data/hard_task_exp_data.json


## check t30_dp

In [26]:
# check t30_dp.json
t30_dp_path = DATA_DIR / "t30_dp.json"
t30_dp = read_json(t30_dp_path)

In [28]:
t30_dp[0].keys()

dict_keys(['task_id', 'prompt', 'test', 'public_test_cases', 'metadata', 'source', 'shards', 'task'])